**First run test plan**

0. Add/patch apply_variant in ecoli/variants/gene_knockout.py


The 5 successful variant/seed pairs avoid this because they never reach a timestep where `mRNA_unique_index` and the ribosome-derived `unique_mRNA_index_ribosomes` are both empty. In other words, there is still at least one full, translatable mRNA left for the listener to map, so `bulk_name_to_idx(...)` has valid inputs.
1. Dry-run: create variant pickles only (use create_variants.py)
```
python runscripts/create_variants.py --config configs/N_gene_knockout_test.json --kb out/all_media_conditions1/parca/kb -o out/test_variants
```

2. Check out/test_variants/metadata.json and open one .cPickle to confirm sim_data.genetic_perturbations or adjusted arrays were set.

3. Run full workflow (will launch Nextflow)
(If Nextflow/containers not configured, run on a machine with Nextflow installed. Use --resume to restart.)
```
python runscripts/workflow.py --config configs/N_gene_knockout_test.json
```

4. Analyze with gene_screen.py / gene_expression_trace.py after sims finish:
(or run gene_expression_trace.py for a specific gene)
```
python reading/gene_screen.py --project gene_knockout_test --variants 1 2 --generations 1 --gene-list /user/home/il22158/work/vEcoli/reading/results/knockout_experiment/knocked_out_test.txt
```

**Second run test plan**
1. Same procedure as the first run test step 3 and 4, with the problematic gene list (p_lilst) from previous simulation.
2. Run gene screen with the p_list.
```bash
python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants $(seq 1 52) --gene-list /user/home/il22158/work/vEcoli/surrogate/results/failure/outlier_genes_all_unique.txt
```

## Variant creation 2 knockouts test

In [11]:
import pickle
import numpy as np
import os

base_dir = '/user/home/il22158/work/vEcoli/out/test_variants'

with open(os.path.join(base_dir, '0.cPickle'), 'rb') as f:
    sd_v0 = pickle.load(f)
with open(os.path.join(base_dir, '1.cPickle'), 'rb') as f:
    sd_v1 = pickle.load(f)
with open(os.path.join(base_dir, '2.cPickle'), 'rb') as f:
    sd_v2 = pickle.load(f)

transcription_v0 = sd_v0.process.transcription
transcription_v1 = sd_v1.process.transcription
transcription_v2 = sd_v2.process.transcription

def analyze_variant(v0, v1, variant_name):
    """Compare two transcription objects and return metrics"""
    print("="*100)
    print(f"PARAMETER COMPARISON: {variant_name}")
    print("="*100)

    metrics = {}

    # ============================================================================
    # 1. rna_synth_prob
    # ============================================================================
    print("\n1. rna_synth_prob (dict of arrays)")
    print("-" * 100)

    rna_synth_prob_v0 = v0.rna_synth_prob
    rna_synth_prob_v1 = v1.rna_synth_prob

    keys_v0 = set(rna_synth_prob_v0.keys())
    keys_v1 = set(rna_synth_prob_v1.keys())
    keys_identical = keys_v0 == keys_v1
    print(f"Keys identical?: {keys_identical}")

    max_diff_across_all = 0
    for key in rna_synth_prob_v0.keys():
        arr_v0 = rna_synth_prob_v0[key]
        arr_v1 = rna_synth_prob_v1[key]
        diff = np.abs(arr_v0 - arr_v1)
        max_diff_across_all = max(max_diff_across_all, np.max(diff))

    print(f"Max value difference across all arrays: {max_diff_across_all:.2e}")

    basal_sum_v0 = np.sum(rna_synth_prob_v0['basal'])
    basal_sum_v1 = np.sum(rna_synth_prob_v1['basal'])
    print(f"Sum of 'basal' array: V0={basal_sum_v0:.6f}, V1={basal_sum_v1:.6f}")
    
    metrics['rna_synth_prob'] = {'max_diff': max_diff_across_all, 'sum': basal_sum_v1}

    # ============================================================================
    # 2. rna_expression
    # ============================================================================
    print("\n2. rna_expression (dict of arrays)")
    print("-" * 100)

    rna_expression_v0 = v0.rna_expression
    rna_expression_v1 = v1.rna_expression

    keys_v0 = set(rna_expression_v0.keys())
    keys_v1 = set(rna_expression_v1.keys())
    keys_identical = keys_v0 == keys_v1
    print(f"Keys identical?: {keys_identical}")

    max_diff_across_all = 0
    for key in rna_expression_v0.keys():
        arr_v0 = rna_expression_v0[key]
        arr_v1 = rna_expression_v1[key]
        diff = np.abs(arr_v0 - arr_v1)
        max_diff_across_all = max(max_diff_across_all, np.max(diff))

    print(f"Max value difference across all arrays: {max_diff_across_all:.2e}")

    basal_sum_v0 = np.sum(rna_expression_v0['basal'])
    basal_sum_v1 = np.sum(rna_expression_v1['basal'])
    print(f"Sum of 'basal' array: V0={basal_sum_v0:.6f}, V1={basal_sum_v1:.6f}")
    
    metrics['rna_expression'] = {'max_diff': max_diff_across_all, 'sum': basal_sum_v1}

    # ============================================================================
    # 3. exp_free
    # ============================================================================
    print("\n3. exp_free (1D array)")
    print("-" * 100)

    exp_free_v0 = v0.exp_free
    exp_free_v1 = v1.exp_free

    shape_v0 = exp_free_v0.shape
    shape_v1 = exp_free_v1.shape
    print(f"Shape identical?: {shape_v0 == shape_v1}")

    diff = np.abs(exp_free_v0 - exp_free_v1)
    max_diff = np.max(diff)
    print(f"Max value difference: {max_diff:.2e}")

    sum_v0 = np.sum(exp_free_v0)
    sum_v1 = np.sum(exp_free_v1)
    print(f"Sum of array: V0={sum_v0:.10f}, V1={sum_v1:.10f}")
    
    metrics['exp_free'] = {'max_diff': max_diff, 'sum': sum_v1}

    # ============================================================================
    # 4. exp_ppgpp
    # ============================================================================
    print("\n4. exp_ppgpp (1D array)")
    print("-" * 100)

    exp_ppgpp_v0 = v0.exp_ppgpp
    exp_ppgpp_v1 = v1.exp_ppgpp

    shape_v0 = exp_ppgpp_v0.shape
    shape_v1 = exp_ppgpp_v1.shape
    print(f"Shape identical?: {shape_v0 == shape_v1}")

    diff = np.abs(exp_ppgpp_v0 - exp_ppgpp_v1)
    max_diff = np.max(diff)
    print(f"Max value difference: {max_diff:.2e}")

    sum_v0 = np.sum(exp_ppgpp_v0)
    sum_v1 = np.sum(exp_ppgpp_v1)
    print(f"Sum of array: V0={sum_v0:.10f}, V1={sum_v1:.10f}")
    
    metrics['exp_ppgpp'] = {'max_diff': max_diff, 'sum': sum_v1}
    
    return metrics

# ============================================================================
# Run analysis for Variant 1 (EG10109 KO)
# ============================================================================
metrics_v1 = analyze_variant(transcription_v0, transcription_v1, "Variant 1 (EG10109 KO) vs Baseline")

# ============================================================================
# Run analysis for Variant 2 (EG10196 KO)
# ============================================================================
print("\n\n")
metrics_v2 = analyze_variant(transcription_v0, transcription_v2, "Variant 2 (EG10196 KO) vs Baseline")

# ============================================================================
# Summary Table - Both Variants Side-by-Side
# ============================================================================
print("\n\n" + "="*130)
print("FINAL TABLE: Parameter Changes After Gene Knockout (Both Variants)")
print("="*130)
print(f"{'Parameter':<20} {'Variant 1 (EG10109 KO)':<55} {'Variant 2 (EG10196 KO)':<55}")
print(f"{'':20} {'Structure':<15} {'Values':<15} {'Sum':<15} {'Structure':<15} {'Values':<15} {'Sum':<15}")
print("-" * 130)

params = ['rna_synth_prob', 'rna_expression', 'exp_free', 'exp_ppgpp']
structures = {
    'rna_synth_prob': '✓ Keys identical',
    'rna_expression': '✓ Keys identical',
    'exp_free': '✓ Shape (3277,)',
    'exp_ppgpp': '✓ Shape (3277,)'
}

for param in params:
    v1_max_diff = metrics_v1[param]['max_diff']
    v1_sum = metrics_v1[param]['sum']
    v2_max_diff = metrics_v2[param]['max_diff']
    v2_sum = metrics_v2[param]['sum']
    
    struct = structures[param]
    v1_values = f"✗ Δ {v1_max_diff:.2e}"
    v2_values = f"✗ Δ {v2_max_diff:.2e}"
    v1_sum_str = f"✓ {v1_sum:.6f}"
    v2_sum_str = f"✓ {v2_sum:.6f}"
    
    print(f"{param:<20} {struct:<15} {v1_values:<15} {v1_sum_str:<15} {struct:<15} {v2_values:<15} {v2_sum_str:<15}")

print("="*130)


PARAMETER COMPARISON: Variant 1 (EG10109 KO) vs Baseline

1. rna_synth_prob (dict of arrays)
----------------------------------------------------------------------------------------------------
Keys identical?: True
Max value difference across all arrays: 2.73e-05
Sum of 'basal' array: V0=1.000000, V1=1.000000

2. rna_expression (dict of arrays)
----------------------------------------------------------------------------------------------------
Keys identical?: True
Max value difference across all arrays: 3.85e-06
Sum of 'basal' array: V0=1.000000, V1=1.000000

3. exp_free (1D array)
----------------------------------------------------------------------------------------------------
Shape identical?: True
Max value difference: 1.51e-06
Sum of array: V0=1.0000000000, V1=1.0000000000

4. exp_ppgpp (1D array)
----------------------------------------------------------------------------------------------------
Shape identical?: True
Max value difference: 1.89e-06
Sum of array: V0=1.00000000

- Conclusion:
    The modification of gene expression and systematic renormalization work at the variant creation process.

## 8 Generations 2 Knockouts test

- Inputs:
    Baseline - no knockouts;
    Variant 1 - knockout gene EG10109 (Gene 1);
    Variant 2 - knockout gene EG10196 (Gene 2).

- Results:
    Gene screen for two knockout genes saved in:
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_0
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_1
        /user/home/il22158/work/vEcoli/reading/results/gene_activity_screen/gene_knockout_test2_v_2

    Where we can see the results matches with expectation - as the two genes both expressed (translated) in baseline, while only Gene 2 expressed in Varaint 1 and only Gene 1 expressed in Variant 2.

    Meanwhile, the cell fate also matched with the early simulation (cutomised knockout function) - Variant 1 succeed to generation 8 and Variant 2 doesn't.

    The main difference between tested knockouts and early simulation is that the Variant 2 cell dies sooner - it only succeeds to 3rd generation while the early simulation succeeds to 6th generation.

## Read outputs in p_list KO sim

**Plan for p_list KO Simulation Analysis**

1. **Run multi-seed gene screening** (seeds 100 and 101 across 52 variants)
   - Command: `python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants $(seq 1 52) --gene-list outlier_genes_all_unique.txt`
   - Expected output structure: `/out/gene_knockout_p_list/{project}/{variant}/{lineage_seed}/`
   - Outputs: CSV files with gene activity, transcription/translation traces, and plots

2. **Aggregate results** - Run aggregation script after screening completes
   - Command: `python reading/aggregate_knockout_results.py --project gene_knockout_p_list`
   - Optional: add `--output-dir`, `--output-file`, or `--base-path` to override defaults
   - Purpose: Consolidate all 52 variants × 2 seeds results into a single summary CSV
   - Input: Success files from `/out/gene_knockout_p_list/success/experiment_id=gene_knockout_p_list/...`
   - Output (default): `/user/home/il22158/work/vEcoli/surrogate/results/gene_knockout_p_list_success_summary.csv`
   - Format: Similar to existing `ko_runs_with_success_generation.csv` with columns:
     * project, variant, lineage_seed, max_success_generation, duration_sec, final_dry_mass, growth_rate, etc.

3. **Validation & analysis**
   - Verify no figure overlap between seeds
   - Check all 52 variants processed with both seeds (expected: 104 result rows)
   - Confirm CSV format matches reference standard
   - Compare seed 100 vs seed 101 results statistically


## Debug

**Investigation Summary**

The knockout definition itself looks reasonable. In [ecoli/variants/gene_knockout.py](/user/home/il22158/work/vEcoli/ecoli/variants/gene_knockout.py), `apply_variant()` maps each `genes_to_knockout` entry to cistron/RNA indices and calls `sim_data.adjust_final_expression(...)` only when it finds a match. The attached config [configs/N_gene_knockout_p_list.json](/user/work/il22158/vEcoli/configs/N_gene_knockout_p_list.json) is also consistent with that behavior: it applies the `gene_knockout` variant to a 52-gene list and runs two seeds (`100` and `101`).

The failure is downstream in the simulation listener, not in the variant generator. A failed task in the Nextflow work directory showed this traceback from [ecoli/processes/listeners/ribosome_data.py](/user/home/il22158/work/vEcoli/ecoli/processes/listeners/ribosome_data.py):

`IndexError: cannot do a non-empty take from an empty axes.`

That happens at the `bulk_name_to_idx(...)` call where `reduced_to_normal_mRNA_indices` is built. For the failing variants/seeds, the mRNA index arrays are empty enough that `np.take(...)` cannot proceed.

The aggregation output [surrogate/results/gene_knockout_p_list_success_summary.csv](/user/home/il22158/work/vEcoli/surrogate/results/gene_knockout_p_list_success_summary.csv) confirms this is not a summary bug: it contains only 10 successful rows, meaning 5 variant/seed combinations completed successfully while the rest failed earlier in the workflow.

The successful variants do not hit this because they still leave at least one full, translatable mRNA in the listener state when `ribosome_data.py` runs. In that case `mRNA_unique_index` and `unique_mRNA_index_ribosomes` are non-empty, so `bulk_name_to_idx(...)` has valid inputs and never reaches the empty-axis path.

Potential fix: Don't change [ecoli/variants/gene_knockout.py](/user/home/il22158/work/vEcoli/ecoli/variants/gene_knockout.py) as the primary fix. The safer change is to harden [ecoli/processes/listeners/ribosome_data.py](/user/home/il22158/work/vEcoli/ecoli/processes/listeners/ribosome_data.py) against empty mRNA/ribosome index arrays and return zero-filled listener values instead of raising. A small defensive check there should prevent the crash for knockout cases that leave no valid mRNA targets.



## Comparison Run: sucessful run gene list

## Third Round Test: gene knockout list for variants that reached generation 8

**To-do plan**

1. Prepare the third-round knockout input from the successful generation-8 gene set.
2. Run the simulation with renewed configuration.
3. Run the knockout screening with the same project structure and the new gene list file.
   - Command: `python reading/gene_screen.py --project gene_knockout_p_list --lineage-seed 100 101 --variants $(seq 1 52) --gene-list surrogate/third_round_tested_gene_list.txt`
4. Aggregate the results and confirm which variants reach generation 8 again.
5. Compare this round against the previous run to see whether the success pattern is reproducible.

**The tested gene list**

They are extracted as the first 50 gene knockouts which succeed to 8th generation in previous simulation (with customised KO function).

Link: [third_round_tested_gene_list.txt](surrogate/results/third_round_tested_gene_list.txt).